In [0]:
use catalog identifier(:catalog);
create or replace table bronze.stations as select *,_metadata.file_path as _source_file,current_timestamp as _ingested_at from read_files('/Volumes/'||:catalog||'/landing/raw/*/stations/',inferSchema=>'false')

In [0]:
create or replace table bronze.tariffs as select *,_metadata.file_path as _source_file,current_timestamp as _ingested_at from read_files('/Volumes/'||:catalog||'/landing/raw/*/tariffs/',inferSchema=>'false')

In [0]:
create or replace table bronze.chargers as
select *,
  _metadata.file_path as _source_file,
  current_timestamp as _ingested_at
from read_files('/Volumes/'||:catalog||'/landing/raw/*/chargers/', format => 'json', multiLine => 'true',inferSchema=>'false')

In [0]:
create table if not exists bronze.members (
  member_id string,
  full_name string,
  email string,
  phone string,
  city string,
  plan string,
  joined_on string,
  change_effective_date string
);


In [0]:
%python
catalog = dbutils.widgets.get("catalog")
spark.sql(f"""
  COPY INTO {catalog}.bronze.members
  FROM '/Volumes/{catalog}/landing/raw/*/members/'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header'='true','inferSchema'='false','mergeSchema'='true')
  COPY_OPTIONS ('mergeSchema'='true')
""")

In [0]:
%python
from pyspark.sql.functions import col, current_timestamp
catalog = dbutils.widgets.get("catalog")
source_path = f"/Volumes/{catalog}/landing/raw/*/sessions/"
checkpoint = f"/Volumes/{catalog}/ops/pipeline_state/sessions"
schema_state = f"/Volumes/{catalog}/ops/pipeline_state/sessions_schema"
sessions_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_state)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file"),
        current_timestamp().alias("_ingested_at"),
    )
)

query = (
    sessions_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.sessions")
)

query.awaitTermination()